# Make a table of the data 

THis script pulls from the outputs of ghcnd_station_download_v1.0 and ENSO_indicies_download_v1.0  
Places station obs a rows and taget variables + indicies as columns

This script take daily observation data from station_data/ghcnd/all_station_data.csv  
and index data from indicies_data/first_pass/enso_historical_indicies.csv  
Then makes a built-out version of the table at the bottom of this page-> https://www.dnr.state.mn.us/climate/journal/el-nino-2026.html  

Note: averaging stations gives a 'station average' not necesarilly a 'Minnesota average'  
    more stations will decrease the difference between the two  

v0.0 designates this is a work in progress  
v1.0 should contribute to a reproducible workflow on any machine with the other 1.0 notebooks


In [2]:
import pandas as pd
from pathlib import Path
import nbformat

In [3]:
# uncomment and run this cell to make data if needed
# %run enso_indicies_download_v1.0.ipynb
# %run ghcnd_station_download_v1.0.ipynb

## Preprocessing
The code below make a raw data table that combines station data and ENSO indicies data.  
It also filters out rows with no observations. 

In [3]:
#assign data to vars for pandas
indicies_data=Path('../indicies_data/first_pass/enso_historical_indicies.csv')
station_data=Path("../station_data/ghcnd/all_station_data.csv")

#read data into the dataframe

#el nino index vals
indicies_data_df=pd.read_csv(
    indicies_data,
    skiprows=0,
    parse_dates=['Date']
)

#staion vals
stations_data_df=pd.read_csv(
    station_data,
    skiprows=0,
    parse_dates=['time']
)

#discard rows with no obs
stations_data_df=stations_data_df.dropna(
    subset=["TMAX", "TMIN", "PRCP", "MDPR", "DAPR",
               "SNOW", "MDSF", "DASF", "SNWD", "WESD", "WESF"],
    how='all'
)

In [4]:
#create a key they can be joined on
indicies_data_df['Month']=indicies_data_df['Date'].dt.to_period('M')
stations_data_df['Month']=stations_data_df['time'].dt.to_period('M')

In [5]:
#join the tables by date

station_obs_enso_indicies= pd.merge(
    left=stations_data_df,
    right=indicies_data_df,
    on='Month',
    how='left',
    validate='many_to_one'
)

In [6]:
#BEHOLD
store_table=Path('../outputs')
store_table.mkdir(parents=True, exist_ok=True)
station_obs_enso_indicies.to_csv(store_table/'station_obs_enso_indicies.csv', index=False)
display(station_obs_enso_indicies)

,station,time,TMAX,TMAX_q,TMIN,TMIN_q,PRCP,PRCP_q,MDPR,MDPR_q,...,lat,lon,name,winter_year,Month,Unnamed: 0,Date,ONI,RONI,TNI
0,USW00014926,1904-09-01,NaN,False,NaN,False,0.0,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,0.68,1.04,0.584
1,USW00014926,1904-09-02,NaN,False,NaN,False,41.4,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,0.68,1.04,0.584
2,USW00014926,1904-09-03,NaN,False,NaN,False,48.5,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,0.68,1.04,0.584
3,USW00014926,1904-09-04,NaN,False,NaN,False,0.0,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,0.68,1.04,0.584
4,USW00014926,1904-09-05,NaN,False,NaN,False,0.0,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,0.68,1.04,0.584
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298952,USW00014920,2026-09-23,20.6,False,6.7,False,0.0,False,NaN,False,...,43.87922,-91.25301,"LA CROSSE REGIONAL AIRPORT, WI US",2027,2026-09,2120,2026-09-01,NaN,NaN,NaN
298953,USW00014920,2026-09-24,21.1,False,9.4,False,0.3,False,NaN,False,...,43.87922,-91.25301,"LA CROSSE REGIONAL AIRPORT, WI US",2027,2026-09,2120,2026-09-01,NaN,NaN,NaN
298954,USW00014920,2026-09-25,22.8,False,15.0,False,0.0,False,NaN,False,...,43.87922,-91.25301,"LA CROSSE REGIONAL AIRPORT, WI US",2027,2026-09,2120,2026-09-01,NaN,NaN,NaN
298955,USW00014920,2026-09-26,22.2,False,11.7,False,0.0,False,NaN,False,...,43.87922,-91.25301,"LA CROSSE REGIONAL AIRPORT, WI US",2027,2026-09,2120,2026-09-01,NaN,NaN,NaN


## Processing

The code below makes the final output of ENSO Proj Phase One


In [7]:
#pull and read data table formed above into a dataframe

raw_table=Path(r"C:\Users\wesja\Desktop\MCAP\MCAP_ENSO_PROJECT\outputs\station_obs_enso_indicies.csv")
raw_table_df=pd.read_csv(raw_table)

In [ ]:
#first replace index values with Strong, Weak, etc
#starts at dthe highest bounds and filters downward

#helper functions
def oni_classify(index):
    if isinstance(index, str):
        return index
    elif pd.isna(index):
        return index
    elif index >= 0.5:
        return 'El Niño'
    elif index < 0.5 and index > -0.5:
        return 'Neutral'
    else:
        return 'La Niña'

def roni_classify(index):
    if isinstance(index, str):
        return index
    elif pd.isna(index):
        return index
    elif index >= 2.0:
        return 'Very Strong El Niño'
    elif  1.5<=index<2.0:
        return 'Strong El Niño'
    elif 1.0<=index<1.5:
        return 'Moderate El Niño'
    elif 0.5<=index<1.0:
        return 'Weak El Niño'
    elif -0.5<index<0.5:
        return 'Neutral Phase'
    elif -0.5>=index>-1.0:
        return 'Weak La Niña'
    elif -1.0>=index>-1.5:
        return 'Moderate La Niña'
    elif -1.5>=index>-2.0:
        return 'Strong La Niña'
    elif index <= -2.0:
        return 'Very Strong La Niña'

#create columns and classify with helper functions
raw_table_df['ONI']=raw_table_df['ONI'].apply(oni_classify)
raw_table_df['RONI']=raw_table_df['RONI'].apply(roni_classify)

#uncomment to check out the new columns
#display(raw_table_df)


,station,time,TMAX,TMAX_q,TMIN,TMIN_q,PRCP,PRCP_q,MDPR,MDPR_q,...,lat,lon,name,winter_year,Month,Unnamed: 0,Date,ONI,RONI,TNI
0,USW00014926,1904-09-01,NaN,False,NaN,False,0.0,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,El Niño,Moderate El Niño,0.584
1,USW00014926,1904-09-02,NaN,False,NaN,False,41.4,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,El Niño,Moderate El Niño,0.584
2,USW00014926,1904-09-03,NaN,False,NaN,False,48.5,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,El Niño,Moderate El Niño,0.584
3,USW00014926,1904-09-04,NaN,False,NaN,False,0.0,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,El Niño,Moderate El Niño,0.584
4,USW00014926,1904-09-05,NaN,False,NaN,False,0.0,False,NaN,False,...,45.54416,-94.05160,"ST. CLOUD REGIONAL AIRPORT, MN US",1905,1904-09,656,1904-09-01,El Niño,Moderate El Niño,0.584
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298952,USW00014920,2026-09-23,20.6,False,6.7,False,0.0,False,NaN,False,...,43.87922,-91.25301,"LA CROSSE REGIONAL AIRPORT, WI US",2027,2026-09,2120,2026-09-01,NaN,NaN,NaN
298953,USW00014920,2026-09-24,21.1,False,9.4,False,0.3,False,NaN,False,...,43.87922,-91.25301,"LA CROSSE REGIONAL AIRPORT, WI US",2027,2026-09,2120,2026-09-01,NaN,NaN,NaN
298954,USW00014920,2026-09-25,22.8,False,15.0,False,0.0,False,NaN,False,...,43.87922,-91.25301,"LA CROSSE REGIONAL AIRPORT, WI US",2027,2026-09,2120,2026-09-01,NaN,NaN,NaN
298955,USW00014920,2026-09-26,22.2,False,11.7,False,0.0,False,NaN,False,...,43.87922,-91.25301,"LA CROSSE REGIONAL AIRPORT, WI US",2027,2026-09,2120,2026-09-01,NaN,NaN,NaN


In [34]:
#select djf obs only,
#group obs by year,
#separate nino and no nino months,
#and group stations


#helper functions
def winter_year(date):
   if date.month==12:
       return date.year+1
   else:
       return date.year

def identify_ninos(df):
    return df[
        (df['ONI'].notna() 
            & (df['ONI']=='El Niño'))
            & (df['RONI'].notna())
            & df['RONI'].isin([
            'Weak El Niño',
            'Moderate El Niño',
            'Strong El Niño',
            'Very Strong El Niño'
        ])
]

def indentify_non_ninos(df):
    return df[
        (df['ONI'].notna() 
            & (df['ONI'] != 'El Niño'))
            & (df['RONI'].notna())
            & ~df['RONI'].isin([
            'Weak El Niño',
            'Moderate El Niño',
            'Strong El Niño',
            'Very Strong El Niño'
        ])
]


#first store djf obs only
raw_table_df['time']=pd.to_datetime(raw_table_df['time'])
djf_obs = raw_table_df[raw_table_df['time'].dt.month.isin([12, 1, 2])].copy()

#creates the winter year column in the dataframe
djf_obs['winter_year']=djf_obs['time'].apply(winter_year)


djf_obs.groupby('station')['PRCP'].sum()







#find all nino rows
#yes_nino=identify_ninos(djf_obs)

#find all non el nino rows
#no_nino=indentify_non_ninos(djf_obs)



#average seasonal precip in inches


#sum seasonal snowfall in inches

#average seasonal temperature

#sum of el nino varities (strong nuetral etc)

#the rows will be the el nino varieties (grouping strong and very strong into one)



station
USW00014913    6839.2
USW00014914    4213.0
USW00014916    3863.3
USW00014918    6501.4
USW00014920    7396.0
USW00014922    6247.5
USW00014925    7744.6
USW00014926    5192.5
USW00014944    5238.2
Name: PRCP, dtype: float64